In [1]:
# 1. Принудительно удаляем старые/сломанные версии
!pip uninstall -y transformers qwen-vl-utils

# 2. Ставим чистые актуальные версии (версия transformers 4.46.1+ стабильно работает с Qwen)
!pip install -q transformers==4.46.1 qwen-vl-utils accelerate peft bitsandbytes torchvision av

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 56.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.7 MB/s eta 0:00:00


In [ ]:
"""
Выпускная квалификационная работа (ВКР)
Тема: Автоматическое распознавание и анализ древнегерманских рунических текстов
Программа: Компьютерная лингвистика, НИУ ВШЭ, 2026

Модуль: Пайплайн эффективного мультимодального дообучения (QLoRA)
        модели Qwen2-VL-7B-Instruct для распознавания (OCR/HTR) рунических знаков.
"""
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import os
import re
import glob
import gc
import zipfile
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor, 
    BitsAndBytesConfig, 
    Trainer,
    TrainingArguments, 
    Qwen2VLForConditionalGeneration
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

# =======================================================================
# 1. ГЛОБАЛЬНАЯ КОНФИГУРАЦИЯ И ГИПЕРПАРАМЕТРЫ
# =======================================================================
OUTPUT_DIR = "/kaggle/working/qwen2vl-7b-runic"
MODEL_ID   = "Qwen/Qwen2-VL-7B-Instruct"
TARGET_COL = "translit"

# Ограничение визуальных токенов для предотвращения OOM (512x512 пикселей)
MAX_PIXELS = 512 * 512  

# Оптимизированные параметры обучения под Kaggle GPU 환경
EPOCHS = 3
BATCH_SIZE = 1
GRADIENT_ACC_STEPS = 16  # Эффективный размер батча = 16
LEARNING_RATE = 2e-4     # Чуть выше для QLoRA 7B

# Настройка видимости GPU (фиксация на первом ядре для устранения конфликтов разделения памяти)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# =======================================================================
# 2. ПОДГОТОВКА ДАННЫХ И ИНВЕНТАРИЗАЦИЯ КОРПУСА
# =======================================================================
print("--- Процедура инициализации данных ---")
candidates = glob.glob("/kaggle/input/**/syn_*.png", recursive=True)

if candidates:
    paths = [Path(p) for p in candidates]
    print(f"Обнаружено {len(paths)} изображений напрямую в директории input.")
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, (
        "Критическая ошибка: В директории /kaggle/input не обнаружены файлы "
        "формата .png или .zip. Убедитесь, что датасет подключен к сессии."
    )
    print(f"Распаковка архивного контейнера: {Path(zips[0]).name}")
    with zipfile.ZipFile(zips[0], 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/synth")
    paths = list(Path("/kaggle/working/synth").rglob("syn_*.png"))

PNG_INDEX = {p.name: p for p in paths}
assert PNG_INDEX, "Ошибка: Синтетические изображения с префиксом syn_*.png не верифицированы."

# Таблица соответствий Elder Futhark (Unicode U+16A0 — U+16FF) конвенции Rundata
ELDER_FUTHARK_MAP = {
    "ᚠ": "f", "ᚢ": "u", "ᚦ": "þ", "ᚨ": "a", "ᚱ": "r", "ᚲ": "k", "ᚷ": "g", "ᚹ": "w",
    "ᚺ": "h", "ᚾ": "n", "ᛁ": "i", "ᛃ": "j", "ᛇ": "ï", "ᛈ": "p", "ᛉ": "R", "ᛊ": "s",
    "ᛏ": "t", "ᛒ": "b", "ᛖ": "e", "ᛗ": "m", "ᛚ": "l", "ᛜ": "ŋ", "ᛞ": "d", "ᛟ": "o"
}

# Регулярное выражение для парсинга структуры имен файлов
FILENAME_REGEX = re.compile(r"^syn_(\d+)_(.*)_([\u16A0-\u16FF]+)\.png$")
dataset_records = []

for filename in PNG_INDEX:
    match = FILENAME_REGEX.match(filename)
    if not match:
        continue
    runic_sequence = match.group(3)
    # Генерация детерминированной транслитерации (Ground Truth сигнал)
    transliteration = "".join(ELDER_FUTHARK_MAP.get(char, "?") for char in runic_sequence)
    dataset_records.append({
        "filename": filename, 
        "runic": runic_sequence,
        "translit": transliteration
    })

df = pd.DataFrame(dataset_records)

# Стратегия разделения выборки (Split Strategy) на уровне надписей (Inscription-level split)
# Исключает утечку данных (Data Leakage) за счет изоляции уникальных текстовых последовательностей
unique_sequences = df["runic"].drop_duplicates().sample(frac=1.0, random_state=42).tolist()
val_cutoff = int(len(unique_sequences) * 0.05)
val_sequences = set(unique_sequences[:val_cutoff])

df["split"] = df["runic"].map(lambda seq: "val" if seq in val_sequences else "train")
print(f"Статистика распределения выборки: {df['split'].value_counts().to_dict()}")

# =======================================================================
# 3. ИНИЦИАЛИЗАЦИЯ И КВАНТОВАНИЕ СЕТИ (NF4 КОНФИГУРАЦИЯ)
# =======================================================================
print("\n--- Загрузка мультимодального процессора и квантование весов ---")
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS, use_fast=False)
processor.tokenizer.padding_side = "right"  # Фиксация стороны паддинга для авторегрессионного декодера

# Конфигурация 4-битного квантования NF4 (Управление VRAM под лимиты Kaggle)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Квантование по нормальному распределению
    bnb_4bit_use_double_quant=True,     # Вторичное квантование квантованных весов (минус ~0.4 бит на параметр)
    bnb_4bit_compute_dtype=torch.float16 # Вычислительный тип данных для деквантования в слоях Attention
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"                 # Явное закрепление за активным GPU
)

# Подготовка k-bit архитектуры к обучению (замораживание базовых весов, активация градиентов для слоев)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False          # Отключение кэширования ключей/значений при обучении

# =======================================================================
# 4. ИНТЕГРАЦИЯ ЛОКАЛЬНЫХ АДАПТЕРОВ (LoRA)
# =======================================================================
# Таргетирование всех линейных слоев трансформера, включая проекции внимания и MLP блоки
# Позволяет компенсировать потерю точности от 4-бит квантования
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# =======================================================================
# 5. ФОРМИРОВАНИЕ ШАБЛОНОВ И ДАТАСЕТА
# =======================================================================
SYSTEM_PROMPT = (
    "You are an expert runologist OCR system. You read Elder Futhark runic inscriptions "
    "and output ONLY their transliteration in the Rundata convention, with no explanation."
)
USER_PROMPT = "Transliterate the runic inscription in this image."

def format_chat_messages(image_path, answer_text=None):
    """Форматирует входные данные под унифицированную схему чата Qwen2-VL."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user", 
            "content": [
                {"type": "image", "image": image_path, "max_pixels": MAX_PIXELS},
                {"type": "text", "text": USER_PROMPT}
            ]
        }
    ]
    if answer_text is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer_text}]})
    return messages

class RunicDataset(Dataset):
    """Пакетный абстрактный класс для ленивой загрузки эпиграфического датасета."""
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, index):
        row = self.data.iloc[index]
        return {
            "image": str(PNG_INDEX[row["filename"]]), 
            "answer": str(row[TARGET_COL])
        }

# Идентификатор паддинг-токена маски изображения
IMAGE_PAD_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<|image_pad|")

def runic_collate_fn(batch):
    """
    Коллейтор для динамического выравнивания батчей и построения маски потерь (Labels).
    Исключает контекст системного и пользовательского промптов из вычисления Cross-Entropy.
    """
    full_chats = [format_chat_messages(b["image"], b["answer"]) for b in batch]
    formatted_texts = [
        processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=False) 
        for chat in full_chats
    ]
    
    image_inputs, video_inputs = process_vision_info(full_chats)
    
    # Токенизация и генерация тензоров тензорного пространства PyTorch
    encoded_inputs = processor(
        text=formatted_texts, 
        images=image_inputs, 
        videos=video_inputs, 
        padding=True, 
        return_tensors="pt"
    )
    
    labels = encoded_inputs["input_ids"].clone()
    # Маскирование токенов заполнения (Padding) и визуальных токенов (Image Pads) значением -100
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMAGE_PAD_TOKEN_ID] = -100
    
    # Расчет длины промпта (Prompt Masking)
    for i, item in enumerate(batch):
        prompt_only_chat = format_chat_messages(item["image"], None)
        prompt_text = processor.apply_chat_template(prompt_only_chat, tokenize=False, add_generation_prompt=True)
        prompt_image_info, _ = process_vision_info(prompt_only_chat)
        
        prompt_len = processor(
            text=[prompt_text], 
            images=prompt_image_info, 
            return_tensors="pt"
        )["input_ids"].shape[1]
        
        # Зануление градиентного влияния промпта на таргет-функцию
        labels[i, :prompt_len] = -100
        
    encoded_inputs["labels"] = labels
    return encoded_inputs

# =======================================================================
# 6. КОНФИГУРАЦИЯ ТРЕНЕРА И ОПТИМИЗАТОРА (TRAINING LOOPS)
# =======================================================================
print("\n--- Инициализация среды и параметров обучения ---")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    fp16=True,                             # Активация смешанной точности на Kaggle Tesla T4/P100
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=10,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    optim="paged_adamw_8bit"                # Выгрузка страниц оптимизатора во избежание OOM
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=runic_collate_fn,
    train_dataset=RunicDataset(df[df.split == "train"]),
    eval_dataset=RunicDataset(df[df.split == "val"])
)

# Очистка мусора и неиспользуемых ссылок ОЗУ/VRAM перед запуском итерации
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nЗапуск итерационного процесса оптимизации параметров сети...")
trainer.train()

# =======================================================================
# 7. ЭКСПОРТ ВЕСОВ И СЕРИАЛИЗАЦИЯ
# =======================================================================
print("\n--- Финализация и сохранение обученного адаптера ---")
adapter_output_path = Path(OUTPUT_DIR) / "lora-adapter"
model.save_pretrained(adapter_output_path)
processor.save_pretrained(adapter_output_path)

print(f"Процесс завершен успешно. Локальный адаптер QLoRA сериализован в: {adapter_output_path}")

2026-05-25 11:55:20.555502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779710120.734934      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779710120.788053      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779710121.185019      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779710121.185063      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779710121.185065      57 computation_placer.cc:177] computation placer alr

--- Процедура инициализации данных ---
Обнаружено 2148 изображений напрямую в директории input.
Статистика распределения выборки: {'train': 2055, 'val': 93}

--- Загрузка мультимодального процессора и квантование весов ---


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 8,331,745,792 || trainable%: 0.4845

--- Инициализация среды и параметров обучения ---

Запуск итерационного процесса оптимизации параметров сети...


Epoch,Training Loss,Validation Loss
0,53.344000,3.949980
1,43.551600,4.290492
